# 08 · Fit SAFT parameters and evaluate predictions

**Requirements:** SAFT/JAX.

Run this notebook from top to bottom in the environment where ThermoWeave is installed. All examples are self-contained unless explicitly marked optional. See the [course index](../tutorials.md) and [theory guide](../theory.md).

<a class="md-button" href="08_parameter_fitting_and_evaluation.ipynb" download>Download notebook</a>


This example fits a **synthetic** unlike dispersion energy. Data generated
 with a known parameter test the fitting machinery; they do not establish
 experimental accuracy or transferability. Real fits need justified bounds,
 uncertainty/weighting, provenance and independent validation data.

In [1]:
import tempfile
from pathlib import Path
import numpy as np
from thermoweave import SAFTgammaMie
from thermoweave.saft import GroupDatabase
from thermoweave.saft.fitting import FitProblem
from thermoweave.evaluation.metrics import mae, rmse
components = {'benzene': {'aCH': 6}, 'heptane': {'CH3': 2, 'CH2': 5}}
true_epsilon = 390.  # epsilon/k_B in K
truth = GroupDatabase.default().copy()
truth.set_unlike_mie('aCH', 'CH2', epsilon=true_epsilon)
generator = SAFTgammaMie(components, database=truth, pressure_Pa=101325.)
Ts = np.array([300., 330., 300.])
X = np.array([[0.2, 0.8], [0.5, 0.5], [0.9, 0.1]])
target = np.array([generator.ln_activity_coefficients(float(t), x) for t, x in zip(Ts, X)])
print('Synthetic ln gamma targets:', target)

Synthetic ln gamma targets: [[0.93530516 0.03661788]
 [0.4091629  0.25163579]
 [0.02913332 1.4490303 ]]


In [2]:
problem = FitProblem(GroupDatabase.default(), components)
problem.add_ln_gamma(('benzene', 'heptane'), Ts, X, target, label='synthetic training data')
problem.vary_mie('aCH', 'CH2', epsilon=(300., 500.))
result = problem.fit(max_nfev=60)
print(result.report())
assert result.success
np.testing.assert_allclose(result.parameters['epsilon[aCH-CH2]'], true_epsilon, rtol=1e-6)

parameter                           start        fitted     std. error
epsilon[aCH-CH2]                      415.64          390     1.94e-09

`gtol` termination condition is satisfied. (cost 2.177e-20, 5 evaluations)

data set                                                 n   RMS (start)     RMS   mean |dev|
synthetic training data                                  6          0.507      0.000   6.05e-11 (ln gamma)


In [3]:
fitted = SAFTgammaMie(components, database=result.database, pressure_Pa=101325.)
holdout_T, holdout_x = 315., [0.35, 0.65]
predicted = fitted.ln_activity_coefficients(holdout_T, holdout_x)
reference = generator.ln_activity_coefficients(holdout_T, holdout_x)
print('Held-out synthetic ln gamma MAE:', mae(predicted, reference))
print('Held-out synthetic ln gamma RMSE:', rmse(predicted, reference))
np.testing.assert_allclose(predicted, reference, atol=1e-7)
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder)/'fitted-pairs.json'
    result.save(path, reference='synthetic tutorial example; not an experimental parameter set')
    print('Saved extension rows:', GroupDatabase.from_json(path).unlike_mie)

Held-out synthetic ln gamma MAE: 4.4651393693584396e-11
Held-out synthetic ln gamma RMSE: 5.427015390181348e-11
Saved extension rows: [{'group_k': 'aCH', 'group_l': 'CH2', 'epsilon': 390.0000000043381, 'lambda_r': None, 'reference': 'synthetic tutorial example; not an experimental parameter set'}]


`result.database` is the full fitted table. `result.save` writes only
 fitted rows as an extension. `GroupDatabase.merged` refuses conflicting existing
 rows: when intentionally replacing a published interaction, use a copied table
 and its explicit setters, or the full fitted database. Nothing here modifies
 the shared default table. See the advanced documentation for neutral
 COSMO-RS corpus evaluation and parameter-space optimization.